# EverFlavor AI - Computer Vision: Food Freshness From Photos

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

> **Status: planning sketch.** The sections, sources, output tables and functions below are the plan.
> The functions are stubs (`NotImplementedError`) and nothing here has been run or tested yet.
> "Run all" is safe: it only runs the setup and prints the plan's status.

---

## Purpose

The project's **computer vision** pipeline (image classification). A later stage: from a photo, say how fresh a fruit or vegetable looks (fresh, getting old, past its best),
and combine that with USDA shelf-life guidance from notebook 04.

This is a different kind of pipeline from notebooks 01-04: the data is **images**, so it needs its own
collection, labeling, splitting and model training, usually with a GPU (Colab: Runtime > Change runtime
type > GPU).

**Safety rule:** the model only ever says how food **looks**. It never says food is safe to eat: spoilage
and harmful bacteria are often invisible. Every answer ends with "check the smell, texture and date; when
in doubt, throw it out" (the USDA advice).

## Inputs and outputs

| | File | Made by |
|---|---|---|
| Input | Public image datasets (section 2; licenses checked first) | Section 2 |
| Input | `data/processed/ingredient_shelf_life.parquet` | Notebook 04 |
| Output | `data/processed/freshness_images.parquet` (index: path, item, state, source, split) | Sections 2-4 |
| Output | `models/freshness/` (the trained model, not stored in GitHub) | Section 5 |

## Contents

1. Setup
2. Image Sources and Licenses
3. One Label Scheme for Every Source
4. Cleaning and Splitting Without Leakage
5. Model Plan
6. Team Decisions and Next Steps

---

## 1. Setup

The same setup as notebooks 01 and 02: it works unchanged in Google Colab, VS Code and Antigravity.

In [ ]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
%pip install -q requests pandas pyarrow pillow imagehash

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# -------------------------------------------------------
# 1. Where is this notebook running? Move to the project folder
#    (the one that holds notebooks/ and data/), as in notebook 01.
# -------------------------------------------------------
try:
    import google.colab  # noqa: F401  # pyright: ignore[reportMissingImports]  (only exists in Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start):
    # Return the nearest folder at or above `start` with notebooks/ and data/, or None
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# -------------------------------------------------------
# 2. The project's code (src/everflavor). If the notebook was opened on its
#    own, the missing files are downloaded from GitHub (temporary name
#    first, __init__.py last), exactly as in notebook 01.
# -------------------------------------------------------
SRC_DIR = PROJECT_ROOT / "src"
EVERFLAVOR_MODULES = ["__init__", "checks", "environment", "charts", "sources", "parsing", "cuisine",
                      "ingredients", "flags", "diets", "nutrition", "features", "pipeline", "review",
                      "recommend", "reporting", "cooking"]
EVERFLAVOR_URL = "https://raw.githubusercontent.com/Chloe-Tu2/The-EverFlavor-AI/main/src/everflavor/"
EVERFLAVOR_DIR = SRC_DIR / "everflavor"
missing_modules = [m for m in EVERFLAVOR_MODULES if not (EVERFLAVOR_DIR / f"{m}.py").exists()]
if missing_modules:
    import urllib.request
    EVERFLAVOR_DIR.mkdir(parents=True, exist_ok=True)
    for module in sorted(missing_modules, key=lambda m: m == "__init__"):
        partial = EVERFLAVOR_DIR / f"{module}.py.part"
        urllib.request.urlretrieve(f"{EVERFLAVOR_URL}{module}.py", partial)
        partial.replace(EVERFLAVOR_DIR / f"{module}.py")
    print(f"Downloaded {len(missing_modules)} project code file(s) (src/everflavor) from GitHub.")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from everflavor.environment import short_path

REFRESH_DOWNLOADS = False   # True downloads every source again instead of using the saved copy
SPLIT_FILES = {s: Path(f"data/processed/recipes_{s}.parquet") for s in ["train", "val", "test"]}
ALL_RECIPES_FILE = Path("data/interim/recipes_all.parquet")
IMAGE_DIR = Path("data/raw/freshness_images")   # images are never committed (too large)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runtime        : {'Google Colab' if IN_COLAB else 'Local Jupyter (VS Code / Antigravity / JupyterLab)'}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
missing_inputs = [str(p) for p in [*SPLIT_FILES.values(), ALL_RECIPES_FILE] if not p.exists()]
if missing_inputs:
    print("\nMISSING notebook 01 outputs:", ", ".join(missing_inputs))
    print("Run notebook 01 first (all of Week 5), then run this notebook again.")
else:
    print("Notebook 01 outputs found.")

---

## 2. Image Sources and Licenses

Candidates to evaluate. **Each license must be read before any download**, and the project may only use
datasets whose license allows it.

| Dataset | What it has | To check |
|---|---|---|
| "Fruits fresh and rotten for classification" (Kaggle) | Apples, bananas, oranges: fresh vs rotten | License, photo conditions (studio vs kitchen) |
| FruitNet (Mendeley Data) | Indian fruits in good / bad / mixed quality | License (listed as CC BY 4.0), item list |
| Food-101 (ETH Zurich) | 101 cooked dishes | Not freshness; useful later for dish recognition |
| Team photos | Produce photographed over several days | Consent, a fixed photo routine, date on each photo |

**Known gap:** most public sets have only two states (fresh / rotten). The "getting old" middle state,
which matters most to a cook, probably needs the team's own photos taken day by day.

In [ ]:
# Planned module: src/everflavor/freshness.py
IMAGE_SOURCES = pd.DataFrame([
    ("kaggle_fresh_rotten", "kaggle", "license to check", "fresh/rotten"),
    ("fruitnet", "mendeley", "CC BY 4.0 (to confirm)", "good/bad/mixed"),
    ("team_photos", "team", "team consent", "day-by-day"),
], columns=["source", "host", "license", "labels"])
print(IMAGE_SOURCES.to_string(index=False))

---

## 3. One Label Scheme for Every Source

Every source labels differently, so each is mapped to one scheme (like cuisine families in notebook 01):

| Column | Values |
|---|---|
| `item` | Our ingredient name (`apple`, `banana`, `tomato` ...) |
| `state` | `fresh`, `aging`, `spoiled` |
| `days_since_purchase` | Only for team photos |
| `source`, `license` | Where the image came from |

In [ ]:
STATE_MAP = {"fresh": "fresh", "good": "fresh", "rotten": "spoiled", "bad": "spoiled", "mixed": "aging"}


def index_images(source: str, folder: Path) -> pd.DataFrame:
    """One row per image: path, item, state (via STATE_MAP), source. Reads labels from folder names."""
    raise NotImplementedError("sketch: image index")

---

## 4. Cleaning and Splitting Without Leakage

- **Duplicates:** near-identical photos (the same fruit from two angles, or a dataset that copies another)
  are found with a perceptual hash and kept in one split only, like `ingredient_group` in notebook 01.
- **Split by source and by photo session**, not by single image, so the test set shows the model new
  fruit, new lighting and new cameras.
- **Balance:** report counts per item and state; a model trained mostly on bananas says little about
  tomatoes.

In [ ]:
def group_near_duplicates(index: pd.DataFrame, max_distance: int = 4) -> pd.Series:
    """Group images whose perceptual hashes (imagehash.phash) differ by at most `max_distance` bits."""
    raise NotImplementedError("sketch: duplicate groups")

---

## 5. Model Plan

Start small: fine-tune a small pretrained image model (for example a MobileNet or EfficientNet from
`torchvision` or `timm`) on `state`, with the item as a second output. Report accuracy **per item** and a
confusion matrix (calling spoiled food fresh is the costly error), and test on team photos taken in a
real kitchen.

---

## 6. Team Decisions and Next Steps

1. Which items first (a short list of common produce).
2. Whether the team takes its own day-by-day photos, and the photo routine.
3. Which datasets pass the license check.

---

## Plan status

What is built, what is a stub, and what needs a team decision first.

In [ ]:
PLAN = pd.DataFrame([
    ("Plan, sources, label scheme", "written"),
    ("2 Image sources", "licenses to check"),
    ("3 Image index", "stub"),
    ("4 Duplicates and split", "stub"),
    ("5 Model", "plan only"),
], columns=["step", "status"])
print(PLAN.to_string(index=False))